# 📈 Detecção de anomalias em sinais vitais

Módulo do sistema de monitoramento multimodal de pacientes.

Recebe uma série temporal de sinais vitais e sinaliza leituras fora do padrão usando
duas abordagens (**z-score** e **Isolation Forest**), avalia contra o *ground truth*
e aplica uma **regra de evolução de prescrição**. A saída padronizada é consumida
pelo dashboard de integração.

> Dados **simulados** (não são de pacientes reais) — limitação assumida por privacidade.

## 1. Dependências

In [ ]:
# No Colab, scikit-learn / pandas / matplotlib já vêm instalados.
# Descomente se rodar num ambiente limpo:
# !pip install -q scikit-learn pandas matplotlib

## 2. Carregar os dados

Tenta ler o CSV do repositório local; se não achar (ex: Colab), baixa direto do GitHub.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV_LOCAL = "../data/vitals/vitais_simulados.csv"
CSV_URL = "https://raw.githubusercontent.com/joaovpires/8iadt-tech-challenge-fase4/main/data/vitals/vitais_simulados.csv"

origem = CSV_LOCAL if os.path.exists(CSV_LOCAL) else CSV_URL
df = pd.read_csv(origem, parse_dates=["timestamp"])
print(f"Origem: {origem}")
print(f"Formato: {df.shape[0]} leituras x {df.shape[1]} colunas")
df.head()

## 3. Exploração rápida

In [ ]:
FEATURES = ["heart_rate", "spo2", "systolic_bp", "diastolic_bp", "temperature", "resp_rate"]

print("Anomalias no ground truth (anomalia_esperada=1):", int(df["anomalia_esperada"].sum()))
df[FEATURES].describe().round(2)

## 4. Abordagem 1 — z-score (baseline univariado)

Marca como anômala qualquer leitura em que algum sinal esteja a mais de 3 desvios-padrão da média.

In [ ]:
z = (df[FEATURES] - df[FEATURES].mean()) / df[FEATURES].std()
df["z_max"] = z.abs().max(axis=1)
df["anomalia_zscore"] = (df["z_max"] > 3).astype(int)

print("Anomalias detectadas (z-score):", int(df["anomalia_zscore"].sum()))

## 5. Abordagem 2 — Isolation Forest (multivariado)

`contamination=0.06` ≈ taxa real de anomalias no dataset (43/720).

In [ ]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(contamination=0.06, random_state=42)
pred = iso.fit_predict(df[FEATURES])  # -1 = anomalia, 1 = normal
df["anomalia_iforest"] = (pred == -1).astype(int)
df["score_iforest"] = -iso.score_samples(df[FEATURES])  # maior = mais anômalo

print("Anomalias detectadas (Isolation Forest):", int(df["anomalia_iforest"].sum()))

## 6. Avaliação contra o ground truth

A coluna `anomalia_esperada` marca as janelas injetadas de propósito (taquicardia, hipóxia, hipertensão).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = df["anomalia_esperada"]
for col in ["anomalia_zscore", "anomalia_iforest"]:
    print("=" * 50)
    print(col)
    print("Matriz de confusão [ [TN FP] [FN TP] ]:")
    print(confusion_matrix(y_true, df[col]))
    print(classification_report(y_true, df[col], digits=3, zero_division=0))

## 7. Visualização das anomalias detectadas

Cada sinal ao longo do tempo, com as leituras marcadas pelo Isolation Forest em vermelho.

In [ ]:
fig, axes = plt.subplots(len(FEATURES), 1, figsize=(13, 15), sharex=True)
anom = df[df["anomalia_iforest"] == 1]
for ax, f in zip(axes, FEATURES):
    ax.plot(df["timestamp"], df[f], lw=0.8, color="steelblue")
    ax.scatter(anom["timestamp"], anom[f], color="red", s=14, zorder=5, label="anomalia")
    ax.set_ylabel(f)
    ax.legend(loc="upper right", fontsize=8)
axes[-1].set_xlabel("timestamp")
fig.suptitle("Sinais vitais com anomalias detectadas (Isolation Forest)", y=1.001)
plt.tight_layout()
plt.show()

## 8. Regra de evolução de prescrição

Alerta se a dose de um medicamento variar mais que um limiar (%) de uma consulta para a outra.
Aqui usamos um histórico fictício de prescrições.

In [ ]:
presc = pd.DataFrame({
    "consulta": pd.to_datetime(["2026-06-01", "2026-06-15", "2026-07-01", "2026-07-15"]),
    "medicamento": ["Losartana"] * 4,
    "dose_mg": [50, 50, 100, 160],
})
presc["variacao_pct"] = (presc["dose_mg"].pct_change() * 100).round(1)

LIMIAR_PCT = 50  # % de variação que dispara alerta
presc["alerta_prescricao"] = presc["variacao_pct"].abs() > LIMIAR_PCT
presc

## 9. Saída padronizada (para a integração)

Função que devolve a lista de alertas no formato consumido pelo dashboard.

In [ ]:
def anomalias_vitais(df, presc, limiar_pct=50):
    """Retorna a lista consolidada de alertas de vitais + prescrição."""
    alertas = []
    for r in df[df["anomalia_iforest"] == 1].itertuples():
        alertas.append({
            "timestamp": str(r.timestamp),
            "tipo": "vital",
            "score": round(float(r.score_iforest), 3),
            "leitura": {f: float(getattr(r, f)) for f in FEATURES},
        })
    for r in presc[presc["variacao_pct"].abs() > limiar_pct].itertuples():
        alertas.append({
            "timestamp": str(r.consulta),
            "tipo": "prescricao",
            "detalhe": f"{r.medicamento}: dose {r.dose_mg}mg ({r.variacao_pct:+.1f}%)",
        })
    return alertas

alertas = anomalias_vitais(df, presc)
print(f"{len(alertas)} alertas gerados")
for a in alertas[:5]:
    print(a)

## 10. Conclusão

- O **Isolation Forest** captura as anomalias multivariadas (queda de SpO₂, picos de FC/pressão)
  com boa precisão/recall contra o ground truth.
- O **z-score** serve de baseline simples e explicável.
- A **regra de prescrição** cobre o requisito de evolução de dosagem.

A função `anomalias_vitais()` entrega os alertas no formato usado pelo alerta consolidado da equipe.